In [1]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import re
import time
import logging
import requests
import xml.etree.ElementTree as ET

from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

from datetime import datetime

import pymupdf
import os

In [3]:
# ==========================================
# CONFIGURATION
# ==========================================

BASE = Path.home() / "Documents/PubMed_Research"
PDF_DIR = BASE / "PDFs"
LOG_DIR = BASE / "Logs"
INPUT = BASE / "pubmed_ids.txt"
REPORT = BASE / "PubMed_Complete_Report.csv"
# EMAIL = "email"
BATCH_SIZE = 100
DELAY = 0.9

for folder in [BASE, PDF_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_DIR / f"{datetime.now():%y%m%d%H%M%S}_download.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

session = requests.Session()
# session.headers.update({
#     "User-Agent": f"PubMedResearchDownloader/2.0 ({EMAIL})"
# })

In [4]:
# config for text extraction
folder_path = "PDFs"
output_folder = "extracted_text"
os.makedirs(output_folder, exist_ok=True)

In [5]:
# ==========================================
# INPUT
# ==========================================


def read_pmids():
    text = INPUT.read_text(encoding="utf-8")
    return list(dict.fromkeys(
        re.findall(r"\b\d{1,10}\b", text)
    ))

In [6]:
# ==========================================
# NCBI ID CONVERSION
# ==========================================


def convert_ids(pmids):
    mapping = {}
    for start in range(0, len(pmids), 200):
        batch = pmids[start:start + 200]
        url = (
            "https://pmc.ncbi.nlm.nih.gov/"
            "tools/idconv/api/v1/articles/"
        )
        
        params = {
            "ids": ",".join(batch),
            "format": "json"
            # "email": EMAIL
        }

        try:
            r = session.get(
                url, params=params, timeout=90
            )
            r.raise_for_status()
            
            for item in r.json().get("records", []):
                pmid = str(item.get("pmid", ""))
                pmcid = item.get("pmcid", "")

                if pmid:
                    mapping[pmid] = {
                        "pmcid": pmcid,
                        "doi": item.get("doi", "")
                    }

        except Exception as e:
            logging.exception("ID conversion failed")
            print("ID conversion error:", e)

        time.sleep(DELAY)

    return mapping

In [7]:
# ==========================================
# PUBMED METADATA
# ==========================================


def get_pubmed_metadata(pmids):
    metadata = {}
    
    for start in range(0, len(pmids), BATCH_SIZE):
        batch = pmids[start:start + BATCH_SIZE]
        url = (
            "https://eutils.ncbi.nlm.nih.gov/"
            "entrez/eutils/efetch.fcgi"
        )
        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml",
            "tool": "PubMedResearchDownloader"
            # "email": EMAIL
        }

        try:
            r = session.get(
                url, params=params, timeout=120
            )
            r.raise_for_status()
            root = ET.fromstring(r.content)

            for article in root.findall(".//PubmedArticle"):

                pmid = article.findtext(
                    ".//PMID", default=""
                )

                title = article.findtext(
                    ".//ArticleTitle", default=""
                )

                journal = article.findtext(
                    ".//Journal/Title", default=""
                )

                year = (
                    article.findtext(
                        ".//PubDate/Year", default=""
                    )
                    or article.findtext(
                        ".//PubDate/MedlineDate", default=""
                    )
                )

                authors = []

                for author in article.findall(
                    ".//AuthorList/Author"
                ):

                    last = author.findtext(
                        "LastName", default=""
                    )

                    fore = author.findtext(
                        "ForeName", default=""
                    )

                    if last:
                        authors.append(
                            f"{last} {fore}".strip()
                        )

                doi = ""

                for aid in article.findall(
                    ".//ArticleIdList/ArticleId"
                ):

                    if aid.attrib.get("IdType") == "doi":
                        doi = aid.text or ""

                metadata[pmid] = {
                    "title": title,
                    "authors": "; ".join(authors),
                    "journal": journal,
                    "year": year,
                    "doi": doi
                }

        except Exception as e:

            logging.exception("PubMed metadata error")

            print("Metadata error:", e)

        time.sleep(DELAY)

    return metadata

In [8]:
# ==========================================
# PMC AWS VERSION DISCOVERY
# ==========================================


def get_pmc_versions(pmcid):

    numeric = pmcid.replace("PMC", "")

    url = "https://pmc-oa-opendata.s3.amazonaws.com/"

    params = {
        "list-type": "2",
        "prefix": f"PMC{numeric}.",
        "delimiter": "/"
    }

    try:

        r = session.get(
            url, params=params, timeout=60
        )

        r.raise_for_status()

        root = ET.fromstring(r.content)

        versions = []

        for item in root.findall(
            ".//{*}CommonPrefixes/{*}Prefix"
        ):

            prefix = item.text or ""

            match = re.search(
                r"(PMC\d+\.\d+)", prefix
            )

            if match:
                versions.append(match.group(1))

        return versions

    except Exception as e:

        logging.exception(
            f"PMC version discovery failed: {pmcid}"
        )

        return []

In [9]:
# ==========================================
# PMC JSON METADATA
# ==========================================


def get_pmc_metadata(version):

    url = (
        "https://pmc-oa-opendata.s3.amazonaws.com/"
        f"metadata/{version}.json"
    )

    try:

        r = session.get(url, timeout=60)

        if r.status_code == 404:
            return None

        r.raise_for_status()

        return r.json()

    except Exception as e:

        logging.exception(
            f"PMC metadata failed: {version}"
        )

        return None

In [10]:
# ==========================================
# PDF DOWNLOAD
# ==========================================
def normalise_pdf_url(pdf_url):

    if not pdf_url:
        return pdf_url

    parsed = urlsplit(pdf_url)

    if parsed.scheme.lower() == "s3":
        return urlunsplit((
            "https",
            f"{parsed.netloc}.s3.amazonaws.com",
            parsed.path,
            parsed.query,
            parsed.fragment
        ))

    return pdf_url

In [11]:
def download_pdf(pmid, version, pdf_url):

    filename = PDF_DIR / f"{pmid}_{version}.pdf"

    if filename.exists():
        return "Already downloaded"

    temporary = filename.with_suffix(".pdf.part")

    try:
        pdf_url = normalise_pdf_url(pdf_url)

        print(f"Downloading PDF from: {pdf_url}")

        with session.get(
            pdf_url,
            stream=True,
            timeout=180
        ) as response:

            response.raise_for_status()

            content_type = response.headers.get("Content-Type", "").lower()

            if "pdf" not in content_type and "octet-stream" not in content_type:
                raise ValueError(
                    f"Unexpected content type: {content_type}"
                )

            with open(temporary, "wb") as pdf_file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 64
                ):
                    if chunk:
                        pdf_file.write(chunk)

        with open(temporary, "rb") as pdf_file:
            if pdf_file.read(5) != b"%PDF-":
                raise ValueError("Downloaded file does not have a valid PDF header")

        temporary.replace(filename)

        return "Downloaded"

    except Exception as e:

        temporary.unlink(missing_ok=True)

        logging.exception(
            f"Download failed: PMID {pmid}, version {version}, URL {pdf_url}"
        )

        print(
            f"Download error for PMID {pmid} "
            f"({version}): {type(e).__name__}: {e}"
        )

        return f"Download error: {type(e).__name__}: {e}"

In [12]:
# Test the S3 URL conversion before running the full download pipeline.
test_url = (
    "s3://pmc-oa-opendata/"
    "PMC10475612.1/PMC10475612.1.pdf"
    "?md5=6056fd47b2519b0369744439646b8f21"
)
print(normalise_pdf_url(test_url))

https://pmc-oa-opendata.s3.amazonaws.com/PMC10475612.1/PMC10475612.1.pdf?md5=6056fd47b2519b0369744439646b8f21


In [13]:
# ==========================================
# EUROPE PMC FREE FULL-TEXT LINKS
# ==========================================


def get_europepmc_links(pmids):

    results = {}

    for start in range(0, len(pmids), 25):

        batch = pmids[start:start + 25]

        query = " OR ".join(
            f"EXT_ID:{pmid}" for pmid in batch
        )

        url = (
            "https://www.ebi.ac.uk/europepmc/"
            "webservices/rest/search"
        )

        params = {
            "query": query,
            "format": "json",
            "resultType": "core",
            "pageSize": 100
        }

        try:

            r = session.get(
                url, params=params, timeout=120
            )

            r.raise_for_status()

            data = r.json()

            for article in data.get(
                "resultList", {}
            ).get("result", []):

                pmid = str(article.get("pmid", ""))

                if not pmid:
                    continue

                links = []

                for item in article.get(
                    "fullTextUrlList", {}
                ).get("fullTextUrl", []):

                    availability = item.get(
                        "availability", ""
                    ).lower()

                    link = item.get("url", "")

                    if availability in ("open access", "free"):

                        if link:
                            links.append(link)

                results[pmid] = "; ".join(
                    dict.fromkeys(links)
                )

        except Exception as e:

            logging.exception(
                "Europe PMC lookup failed"
            )

            print("Europe PMC error:", e)

        time.sleep(DELAY)

    return results

In [14]:
# ==========================================
# MAIN PIPELINE
# ==========================================


def main():

    pmids = read_pmids()

    print("\nTotal unique PMIDs:", len(pmids))

    print("\n1. Retrieving article identifiers...")

    identifiers = convert_ids(pmids)

    print("\n2. Retrieving PubMed metadata...")

    metadata = get_pubmed_metadata(pmids)

    print("\n3. Finding additional free full-text links...")

    external_links = get_europepmc_links(pmids)

    report = []

    print("\n4. Searching PMC AWS for PDFs...\n")

    for number, pmid in enumerate(pmids, 1):

        ids = identifiers.get(pmid, {})

        pmcid = ids.get("pmcid", "")

        article = metadata.get(pmid, {})

        status = "Not found in PMC"

        pdf_count = 0

        if pmcid:

            versions = get_pmc_versions(pmcid)

            if versions:

                candidates = []

                for version in versions:

                    info = get_pmc_metadata(version)

                    if not info:
                        continue

                    pdf_url = info.get("pdf_url", "")

                    if pdf_url:

                        candidates.append({
                            "version": version,
                            "pdf_url": pdf_url,
                            "open_access": info.get(
                                "is_pmc_openaccess", False
                            ),
                            "license": info.get(
                                "license_code", ""
                            )
                        })

                    time.sleep(DELAY)

                # Prefer the PMC Open Access version
                candidates.sort(
                    key=lambda x: bool(x["open_access"]),
                    reverse=True
                )

                if candidates:

                    for candidate in candidates:

                        result = download_pdf(
                            pmid,
                            candidate["version"],
                            candidate["pdf_url"]
                        )

                        if result in (
                            "Downloaded",
                            "Already downloaded"
                        ):

                            status = result

                            pdf_count += 1

                        elif status == "Not found in PMC":

                            status = result

                    if pdf_count:
                        status = "Downloaded"

                else:

                    status = "PMC article without eligible PDF"

        if not pmcid and external_links.get(pmid):

            status = "External free-full-text link available"

        report.append({

            "PMID": pmid,

            "Title": article.get("title", ""),

            "Authors": article.get("authors", ""),

            "Journal": article.get("journal", ""),

            "Year": article.get("year", ""),

            "DOI": article.get(
                "doi", ids.get("doi", "")
            ),

            "PMCID": pmcid,

            "PDF_Status": status,

            "Europe_PMC_Free_Full_Text_Links":
                external_links.get(pmid, ""),

            "PubMed_URL":
                f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",

            "PMC_URL":
                f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
                if pmcid else ""

        })

        print(
            f"[{number}/{len(pmids)}] "
            f"{pmid}: {status}"
        )

        time.sleep(DELAY)

    # ======================================
    # SAVE CSV REPORT
    # ======================================

    fields = list(report[0].keys()) if report else [
        "PMID", "Title", "Authors", "Journal",
        "Year", "DOI", "PMCID", "PDF_Status",
        "Europe_PMC_Free_Full_Text_Links",
        "PubMed_URL", "PMC_URL"
    ]

    with open(
        REPORT,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fields
        )

        writer.writeheader()
        writer.writerows(report)

    downloaded = sum(
        1 for x in report
        if x["PDF_Status"] == "Downloaded"
    )

    already = sum(
        1 for x in report
        if x["PDF_Status"] == "Already downloaded"
    )

    external = sum(
        1 for x in report
        if x["Europe_PMC_Free_Full_Text_Links"]
    )

    print("\n================================")
    print("PUBMED DOWNLOAD FINISHED")
    print("================================")

    print("Total articles:", len(pmids))

    print("Downloaded:", downloaded)

    print("Already downloaded:", already)

    print("External free-text links:", external)

    print("PDF folder:", PDF_DIR)

    print("CSV report:", REPORT)

In [15]:
if __name__ == "__main__":
    main()


Total unique PMIDs: 8

1. Retrieving article identifiers...

2. Retrieving PubMed metadata...

3. Finding additional free full-text links...

4. Searching PMC AWS for PDFs...

[1/8] 37670947: Downloaded
[2/8] 35046949: Downloaded
[3/8] 37509200: Downloaded
[4/8] 37854601: Downloaded
[5/8] 39404403: Downloaded
[6/8] 36547975: Not found in PMC
[7/8] 33326254: Not found in PMC
[8/8] 28813678: PMC article without eligible PDF

PUBMED DOWNLOAD FINISHED
Total articles: 8
Downloaded: 5
Already downloaded: 0
External free-text links: 8
PDF folder: /home/marta/Documents/PubMed_Research/PDFs
CSV report: /home/marta/Documents/PubMed_Research/PubMed_Complete_Report.csv


In [16]:
# extract text from pdfs
for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):

        pdf_path = os.path.join(folder_path, filename)

        doc = pymupdf.open(pdf_path)

        text = "\n".join(page.get_text() for page in doc)

        output_filename = os.path.splitext(filename)[0] + ".txt"
        output_path = os.path.join(output_folder, output_filename)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

        doc.close()

        print(f"Extracted: {filename}")

Extracted: 37854601_PMC10579806.1.pdf
Extracted: 35046949_PMC8761789.1.pdf
Extracted: 37509200_PMC10377354.1.pdf
Extracted: 37670947_PMC10475612.1.pdf
Extracted: 39404403_PMC11476222.1.pdf
